# MECHA in metafspm, 7: the section's hydraulic options

MECHA computes a section's conductances from its anatomy and a few dozen parameters (`doc/equations.rst`,
Conductances; `doc/legacy.rst`, Coverage). This tutorial goes through the options a modeller changes to describe a
root's development or an experiment: the **barrier codes**, the **φ-thickenings**, the **aquaporins**, the
**plasmodesmata**, the **walls**, the **xylem**, and what each does to the section's radial conductivity kr (MECHA's
standard problem: each section alone, the soil at 0 hPa, the xylem at −5000 hPa).

Every option is compared with the legacy MECHA in `test/comparison/` (`test_develop_barriers.py`,
`test_hydraulic_options.py`, `test_plasmodesmata_factors.py`, `test_postprocess.py`). **Run time:** about a minute.

> **Developer tutorial 7** of ten:
> | | notebook | what it covers |
> |---|---|---|
> | 0 | `00_overview.ipynb` | the package's modules, the original MECHA and its reactions branch, how the port is validated, the docs |
> | 1 | `01_granap_mecha.ipynb` | GRANAP's anatomies and MECHA's hydraulics coupled in a plant model; MECHA's inputs by role; scenarios; the barriers |
> | 2 | `02_soil_gradient.ipynb` | the plant problem with a soil drier on one side of a root section; a root partly out of contact, by position or by cells |
> | 3 | `03_needle_solutes.ipynb` | a pine needle of three sections, its transpiration, MechaSolutes over a day |
> | 4 | `04_full_plant.ipynb` | a whole seedling in its soil, run to the fixed point |
> | 5 | `05_extend_the_component.ipynb` | add a variable and a process to the component, run and test it |
> | 6 | `06_solute_storage_and_decay.ipynb` | the solutes' storage fractions, their first-order loss and the liquid boundary |
> | 7 | `07_section_options.ipynb` | the section's hydraulic options: the barrier codes 0-11, φ-thickenings, aquaporins, plasmodesmata rules, walls, xylem pieces, tracheids, measured axial conductances |
> | 8 | `08_standard_problem.ipynb` | the standard problem's boundaries (soil sides, xylem and phloem pressures or flows, the elongation zone), the osmotic scenarios, the flows by layer |
> | 9 | `09_solute_options.ipynb` | the solutes' pathways, schemes and time steps, held concentrations and steady states, the coupled water and sugar |
>
> New to MECHA? Start with the user tutorials (`../user/01_first_run.ipynb`): they solve a root section and explain
> its results without the framework's details.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from openalea.metafspm.coupling.composite_wrapper import CompositeModel
from openalea.metafspm.coupling.translator import Translator
from openalea.metafspm.scene.scene import Scene

from openalea.granap.metafspm.architectures import RootAxis
from openalea.granap.metafspm.component import GranapAnatomy, junction_rules
from openalea.mecha.component import MechaHydraulics


class RootSection(CompositeModel):                     # the first tutorial's plant model
    initiators = (RootAxis, GranapAnatomy)
    from_scale, nodes, wiring = "SubOrgan", "Compartment", junction_rules

    def __init__(self, data_structure, time_step):
        self.anatomy = GranapAnatomy(data_structure=data_structure, time_step=time_step)
        self.hydraulics = MechaHydraulics(data_structure=data_structure, time_step=time_step)
        self.declare_data_and_couple_components(translator_path=Translator(),
                                                components=(self.anatomy, self.hydraulics))

    def run(self):
        self.anatomy()
        self.hydraulics()


def section(anatomy=None, **parameters):
    """A scene of one root section (GRANAP's *anatomy*: its parameters; the monocot root by default), MECHA's
    *parameters* in its scenario; returns (scene, its DataStructure, its MechaHydraulics)."""
    scenario = dict(parameters)
    if anatomy is not None:
        scenario["root_anatomy"] = anatomy
    scene = Scene(pd.DataFrame([dict(plant="root", model=RootSection, x=0., y=0., z=0., rotation=0.,
                                     scenario={"parameters": scenario})]))
    return scene, scene.populations[0].data_structure, scene.populations[0].instance.hydraulics


def solve(scene, barrier=1., **parameters):
    """One step at *barrier* with MECHA's *parameters* changed for it (then restored): the section's kr
    (cm hPa-1 d-1), Kx (cm4 hPa-1 d-1) and standard uptake (cm3 d-1)."""
    ds, mecha = scene.populations[0].data_structure, scene.populations[0].instance.hydraulics
    saved = {name: getattr(mecha, name) for name in parameters}
    for name, value in parameters.items():
        setattr(mecha, name, value)
    ds.set("barrier", np.full(ds.n_nodes(), float(barrier)))
    try:
        scene.run()
    finally:
        for name, value in saved.items():
            setattr(mecha, name, value)
    return pd.Series({"kr": float(ds.get("kr")[0]), "Kx": float(ds.get("Kx")[0]),
                      "uptake": float(ds.get("standard_uptake")[0])})

## 1. The barrier codes

The segment's `barrier` input (role "Development": a development model, or the scenario) is MECHA's maturity code.
Each code sets which walls take which conductivity: the table below gives, per code, the source of each wall type's
`kw` (`conductance._KW_CONFIGS`); a Casparian strip, suberin (inner, outer face) or lignin cuts the wall to
`kw_casparian`, `kw_suberin_in`, `kw_suberin_out` or `kw_lignin` (10⁻¹⁶ by default: impermeable).

In [2]:
from openalea.mecha.conductance import _KW_CONFIGS, _KW_KEYS

pd.DataFrame(_KW_CONFIGS, index=[key.removeprefix("kw_") for key in _KW_KEYS]).T.rename_axis("barrier")

,endo_endo,exo_exo,exo_epi,exo_cortex,cortex_cortex,endo_peri,endo_cortex,passage
barrier,,,,,,,,
0,kw,kw,kw,kw,kw,kw,kw,kw
1,kw_casparian,kw,kw,kw,kw,kw,kw,kw
2,kw_casparian,kw,kw,kw,kw,kw_suberin_in,kw_suberin_out,kw
3,kw_casparian,kw,kw,kw,kw,kw_suberin_in,kw_suberin_out,kw_suberin_in
4,kw_casparian,kw_casparian,kw,kw,kw,kw_suberin_in,kw_suberin_out,kw_suberin_in
5,kw_casparian,kw_casparian,kw,kw,kw,kw,kw,kw
6,kw_casparian,kw_casparian,kw_suberin_out,kw_suberin_in,kw,kw,kw,kw
7,kw,kw_casparian,kw,kw,kw,kw,kw,kw
8,kw_casparian,kw_casparian,kw_suberin_out,kw_suberin_in,kw,kw_suberin_in,kw_suberin_out,kw


- 0: no barrier, the root tip (its water goes to the protophloem: the standard problem needs a phloem condition,
  tutorial 8);
- 1: the endodermis' Casparian strip; 2: + its suberin, with passage cells; 3: fully suberized;
- 4–9: the exodermis' variants (a Casparian strip, suberin, lignin, with the endodermis' or not); 10: the exodermis
  suberized with a Casparian strip (`develop`'s); 11: barrier 8 and every wall of a sclerenchyma cell lignified.

kr for each code on the monocot root:

In [3]:
scene, ds, mecha = section()
codes = pd.DataFrame({barrier: solve(scene, barrier) for barrier in range(1, 12)}).T.rename_axis("barrier")
codes["kr"].plot.bar(logy=True, ylabel="kr (cm hPa⁻¹ d⁻¹)", figsize=(8, 3.5), rot=0)
plt.show()
codes

/tmp/ipykernel_1733890/2981243680.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,kr,Kx,uptake
barrier,,,
1,0.000110,1.194155,0.002772
2,0.000041,1.194155,0.001025
3,0.000041,1.194155,0.001025
4,0.000037,1.194155,0.000942
5,0.000089,1.194155,0.002240
6,0.000064,1.194155,0.001621
7,0.000207,1.194155,0.005204
8,0.000032,1.194155,0.000812
9,0.000072,1.194155,0.001807


**Reading it.** The endodermis' Casparian strip (1) forces the water through the endodermis' membranes; suberin (2,
3) closes them too, leaving the plasmodesmata (and the passage cells, which GRANAP's root has none of: 2 and 3 are
then equal), and kr falls by more than half. The exodermis' barriers add a second ring: with the endodermis' (4, 5,
6, 8, 9, 11) kr falls further; alone (7, a Casparian strip in the exodermis only), the water crosses the exodermis'
membranes but not the endodermis', and kr is the highest; 10 (the exodermis suberized, no endodermal strip) is about
1. With every barrier wall at 10⁻¹⁶, codes that close the same walls give the same kr: give the barriers a measured
conductivity (`kw_suberin_in=1e-9`, for example) to separate them.

Barrier 0 has no xylem boundary: kr is NaN until the phloem is given a potential (tutorial 8).

Barrier 11 lignifies the sclerenchyma's walls. GRANAP's roots have none (its stems and leaves do), so here the
outermost quarter of the cortex cells is retagged sclerenchyma, as a stem's ring would be:

In [4]:
class RootWithSclerenchyma(RootSection):
    """The outermost quarter of the cortex cells (by their distance to the centre) tagged sclerenchyma before MECHA
    reads the section."""
    def run(self):
        self.anatomy()
        ds = self.anatomy.data_structure
        cortex = (ds.get("kind") == 0) & (np.asarray(ds.get("cgroup")) == 4)
        x, y = ds.get("section_x"), ds.get("section_y")
        distance = np.hypot(x - x[cortex].mean(), y - y[cortex].mean())
        tissue = np.asarray(ds.get("tissue"), dtype=object).copy()
        tissue[cortex & (distance > np.quantile(distance[cortex], 0.75))] = "sclerenchyma"
        ds.set("tissue", tissue)
        self.hydraulics()


scler = Scene(pd.DataFrame([dict(plant="root", model=RootWithSclerenchyma, x=0., y=0., z=0., rotation=0.,
                                 scenario={"parameters": {}})]))
pd.DataFrame({barrier: solve(scler, barrier) for barrier in (8, 11)}).T.rename_axis("barrier")

,kr,Kx,uptake
barrier,,,
8,0.000032,1.194155,0.000817
11,0.000030,1.194155,0.000757


With the sclerenchyma's walls lignified (11), the water crosses that ring through its membranes only: kr falls
compared with barrier 8, which has the same exodermis and endodermis.

## 2. φ-thickenings

Some species (Brassicaceae, for example) thicken and lignify the cortex walls of one or a few layers: a φ-thickening.
`n_phi_layers` sets how many layers, `phi_type` where: 1 the inner cortex, 2 the outer, 3 the middle. Their walls
between two cells of the same layer take `kw_lignin`:

In [5]:
phi = pd.DataFrame({f"{layers} layer(s), type {kind}": solve(scene, 1., n_phi_layers=float(layers), phi_type=float(kind))
                    for layers, kind in ((0, 1), (1, 1), (2, 1), (1, 2), (1, 3))}).T
phi

,kr,Kx,uptake
"0 layer(s), type 1",0.000110,1.194155,0.002772
"1 layer(s), type 1",0.000080,1.194155,0.002020
"2 layer(s), type 1",0.000065,1.194155,0.001643
"1 layer(s), type 2",0.000089,1.194155,0.002242
"1 layer(s), type 3",0.000082,1.194155,0.002066


The water still crosses the thickened layer through its cells' membranes and plasmodesmata, so kr falls but stays
of the same order.

## 3. Aquaporins

A membrane's conductivity is `kmb` + the aquaporins' `kaqp` × a tissue factor (`kaqp_epi_factor`,
`kaqp_exo_factor`, `kaqp_cortex_factor`, `kaqp_endo_factor`, `kaqp_stele_factor`). In the cortex, the activity can
also vary across the layers: `ratio_cortex` is the activity at the innermost cortex walls over the outermost (1:
uniform), linear in the distance to the centre, with the cortex's mean kept:

In [6]:
aquaporins = pd.DataFrame({
    "defaults": solve(scene),
    "aquaporins ÷10": solve(scene, kaqp=mecha.kaqp / 10),
    "endodermis ×2": solve(scene, kaqp_endo_factor=2.),
    "cortex ×0.5": solve(scene, kaqp_cortex_factor=0.5),
    "cortex gradient 0.25": solve(scene, ratio_cortex=0.25),
    "cortex gradient 4": solve(scene, ratio_cortex=4.),
}).T
aquaporins

,kr,Kx,uptake
defaults,0.000110,1.194155,0.002772
aquaporins ÷10,0.000045,1.194155,0.001131
endodermis ×2,0.000154,1.194155,0.003875
cortex ×0.5,0.000109,1.194155,0.002736
cortex gradient 0.25,0.000109,1.194155,0.002753
cortex gradient 4,0.000111,1.194155,0.002784


With a Casparian strip, the endodermis' membranes are on the water's only path: their aquaporins matter most. The
cortex gradient changes kr little: the cortex walls carry most of the water around its cells.

Note MECHA's rule for a membrane without aquaporin activity (`kaqp` 0, or a tissue factor 0): it is cut (10⁻¹⁶), as
an air-filled space's, not left at `kmb`. With `kaqp=0` every membrane is cut and kr falls by seven orders of
magnitude; to study the membranes without aquaporins, give `kaqp` a small value instead.

## 4. Plasmodesmata

Each plasmodesma conducts `kpl`; their number per wall is a frequency per interface (`fplxheight_*`) × the wall's
length, × an aperture factor per tissue (`kpl_cortex_factor`, `kpl_endo_in_factor`, `kpl_endo_out_factor`,
`kpl_PPP_factor`, `kpl_PCC_factor`). `plasmodesmata_rules` chooses the rules: 1 the original MECHA's (default), 2
the refactored legacy's (`develop`'s factor map), for the comparisons. The aperture can also depend on the distance to
the soil contact (`pd_spatial_profile`, tutorial 2).

In [7]:
plasmodesmata = pd.DataFrame({
    "defaults": solve(scene, 3.),
    "closed (kpl 0)": solve(scene, 3., kpl=0.),
    "cortex closed": solve(scene, 3., kpl_cortex_factor=0.),
    "endodermis' inner side ×4": solve(scene, 3., kpl_endo_in_factor=4.),
    "the legacy's rules": solve(scene, 3., plasmodesmata_rules=2.),
}).T
plasmodesmata

,kr,Kx,uptake
defaults,4.071697e-05,1.194155,1.024934e-03
closed (kpl 0),9.365522e-14,1.194155,2.357505e-12
cortex closed,1.980671e-13,1.194155,4.985777e-12
endodermis' inner side ×4,6.032479e-05,1.194155,1.518506e-03
the legacy's rules,4.071697e-05,1.194155,1.024934e-03


At barrier 3 (a suberized endodermis) the plasmodesmata are the water's way through the endodermis: closing them
divides kr by orders of magnitude, opening the endodermis' inner side raises it. With every factor at 1, the two
rules agree.

## 5. Walls, septa and air spaces

- `thickness_source`: 2 each wall's own thickness (GRANAP's, default), 1 one value `thickness` for every wall (the
  original MECHA's, 1.5 µm);
- `xylem_pieces`: 1 when the xylem vessels are made of several cells, the walls between two of them septa of
  near-zero conductance;
- `membrane_floor`: an intercellular space's membrane without aquaporins, 1 cut (an air-filled space conducts no
  water across a membrane, default), 2 kept at `kmb` (the reactions branch's and `develop`'s);
- `k_interc`: the intercellular spaces' membranes with barriers (0 by default).

In [8]:
walls = pd.DataFrame({
    "defaults": solve(scene),
    "one thickness, 1.5 µm": solve(scene, thickness_source=1.),
    "xylem pieces": solve(scene, xylem_pieces=1.),
    "air spaces' membranes conducting": solve(scene, membrane_floor=2.),
    "kw ×2": solve(scene, kw=2 * mecha.kw),
}).T
walls

,kr,Kx,uptake
defaults,0.000110,1.194155,0.002772
"one thickness, 1.5 µm",0.000106,1.194155,0.002677
xylem pieces,0.000110,1.194155,0.002772
air spaces' membranes conducting,0.000110,1.194155,0.002772
kw ×2,0.000125,1.194155,0.003144


On this root the vessels are single cells and the air spaces carry no water: `xylem_pieces` and `membrane_floor`
change nothing visible. One thickness instead of each wall's own lowers kr by 4 %.

## 6. The xylem: tracheids and measured conductances

`Kx` (cm⁴ hPa⁻¹ d⁻¹) is the section's axial conductance: the sum of its vessels' Poiseuille conductances × its
height. Where the vessels' conductances are measured (or some are embolized), `measured_axial_conductivity` gives a
conduit its own specific conductance (MECHA's `Kax_source` 2): it replaces Poiseuille's in Kx, in the axial
boundary and between sections in the plant problem. Here half the vessels are embolized (0):

In [9]:
from openalea.mecha.postprocess import VISCOSITY_XYLEM, poiseuille

solve(scene)
vessels = np.flatnonzero(np.isin(ds.get("cgroup"), (13, 19, 20)) & (ds.get("kind") == 0))
area, height = ds.get("cell_area")[vessels] * 1e6, ds.get("length")[vessels] * 1e6          # µm2, µm
own = poiseuille(area, height, VISCOSITY_XYLEM) * height / 1e4                           # their Poiseuille values
measured = np.full(ds.n_nodes(), np.nan)
measured[vessels] = np.where(np.arange(vessels.size) % 2 == 0, own, 0.)
ds.set("measured_axial_conductivity", measured)
embolized = solve(scene)
ds.set("measured_axial_conductivity", np.full(ds.n_nodes(), np.nan))
pd.DataFrame({"Poiseuille": solve(scene), "half the vessels embolized": embolized}).T

,kr,Kx,uptake
Poiseuille,0.000110,1.194155,0.002772
half the vessels embolized,0.000109,0.718160,0.002748


Kx falls by the embolized vessels' share; kr hardly changes: the vessels' boundary (here the xylem potential) is
not what limits the radial flow.

In needles, the **transfusion tracheids** (cgroup 18) are water-filled conduits: `tracheid_as_xylem` 1 gives their
membranes the xylem vessels' (`develop`'s), without the xylem's boundary. GRANAP's needle has them (tutorial 3).

## 7. Monocots, dicots and the cambium

The anatomy is GRANAP's: `{"preset": "for_dicot_root"}` gives a dicot root (fewer cortex layers, a central xylem,
companion cells), the default a monocot (maize-like). The options above apply to both:

In [10]:
dicot, dicot_ds, _ = section(anatomy={"preset": "for_dicot_root"})
pd.DataFrame({"monocot": solve(scene, 3.), "dicot": solve(dicot, 3.)}).T

,kr,Kx,uptake
monocot,0.000041,1.194155,0.001025
dicot,0.000056,2.982853,0.002075


MECHA's cell groups include the cambium (22, `develop`'s: its plasmodesmata frequencies `fplxheight_*_cambium`,
`kpl_cambium_stele_factor`); GRANAP tags its cambium 12 (with the companion cells), so a cambium of 22 comes from
another anatomy model or a retagging (as the sclerenchyma above).

## Try it yourself

1. Give the suberin a conductivity of 10⁻⁹ (`kw_suberin_in`, `kw_suberin_out`): which barrier codes separate?
2. Find the `kaqp_endo_factor` that doubles kr at barrier 1.
3. Plot kr against `n_phi_layers` for each `phi_type`.